# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos: sistema LLM + RAG con información tabular**

* **Nombres y matrículas:**

*   Joel Arturo Becerril Balderas — A01797427
*   Diego Villegas Juarez — A01797347
*   Camilo José López Amaya — A01797343
*   Manuel Alejandro Ambriz Baca — A01686824

* **Número de Equipo:** 25


* ##### **El formato de este cuaderno de Jupyter es libre, pero incluye al menos lo solicitado en el archivo PDF asociado a esta actividad.**

* ##### **Pueden importar los paquetes o librerías que requieran.**

* ##### **Pueden incluir las celdas y líneas de código que deseen.**

In [26]:
# Instalación de dependencias
# groq: cliente oficial para la API de Groq (LLM gratuito)
# python-dotenv: carga variables de entorno desde .env
!pip install pdfplumber sentence-transformers faiss-cpu groq python-dotenv -q


## Descripción del sistema RAG implementado

**¿Qué hace este notebook?**
Construye un chatbot de tipo RAG (Retrieval-Augmented Generation) que extrae información de dos PDFs de referencia (`python_cheatsheet.pdf` y `ml_cheatsheet.pdf`) y responde preguntas usando recuperación semántica vectorial + generación con LLM.

**Pipeline:**
1. Extracción de texto + tablas (vía `pdfplumber`) de ambos PDFs
2. División en chunks con solapamiento para preservar contexto
3. Generación de embeddings semánticos con `all-MiniLM-L6-v2` (Sentence Transformers)
4. Indexado en base de datos vectorial FAISS
5. Recuperación de los *k* fragmentos más relevantes a la pregunta (búsqueda L2)
6. Generación de respuesta contextualizada con **Llama 3.3 70B** vía Groq (API gratuita)

**Conceptos clave:**
- **RAG:** combina búsqueda semántica con generación LLM; reduce alucinaciones anclando la respuesta en documentos reales
- **FAISS (Facebook AI Similarity Search):** índice vectorial para búsqueda de similitud en tiempo sublineal
- **Embeddings:** representaciones vectoriales densas del significado semántico del texto
- **Chunking con overlap:** evita perder contexto en los bordes de fragmentos

**Casos de uso de negocio:**
- Chatbots de soporte técnico basados en documentación interna
- Asistentes de onboarding con manuales corporativos
- Sistemas de Q&A sobre políticas, regulaciones y procedimientos
- Recuperación de conocimiento en repositorios documentales grandes

**Advertencias técnicas:**
- Los PDFs tipo infografía tienen extracción de texto imperfecta por su diseño visual multi-columna
- Se requiere la variable de entorno `GROQ_API_KEY` configurada (disponible en el `.env` del workspace)
- Se usa device MPS para aprovechar el Neural Engine de Apple Silicon M5 (~4-5× vs CPU)


In [27]:
# ── Importaciones y configuración ────────────────────────────────────────────

import os
import numpy as np
import pdfplumber        # extracción de texto y tablas de PDFs
import faiss             # índice vectorial para búsqueda semántica rápida
import torch
from groq import Groq
from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer

# Cargar GROQ_API_KEY desde el .env del workspace
load_dotenv(dotenv_path='/Users/joelbecerril/MNA_WORKSPACE/.env')

# Apple Silicon M5: usar MPS para acelerar embeddings (~4-5× vs CPU)
if torch.backends.mps.is_available():
    device = "mps"
elif torch.cuda.is_available():
    device = "cuda"
else:
    device = "cpu"
print(f"Dispositivo seleccionado: {device}")

BASE_DIR = os.getcwd()
PDF_PATHS = {
    "python_cheatsheet": os.path.join(BASE_DIR, "python_cheatsheet.pdf"),
    "ml_cheatsheet":     os.path.join(BASE_DIR, "ml_cheatsheet.pdf"),
}

for name, path in PDF_PATHS.items():
    status = "✓ encontrado" if os.path.exists(path) else "✗ NO ENCONTRADO"
    print(f"  {name}: {status}")


Dispositivo seleccionado: mps
  python_cheatsheet: ✓ encontrado
  ml_cheatsheet: ✓ encontrado


In [28]:
# ── Extracción de texto y tablas de los PDFs ──────────────────────────────────

def extract_pdf_content(pdf_path: str) -> str:
    """Extrae texto plano + tablas (→ Markdown) de un PDF con pdfplumber."""
    sections = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, start=1):
            # Texto corrido de la página
            text = page.extract_text()
            if text and text.strip():
                sections.append(f"[Página {page_num}]\n{text.strip()}")

            # Tablas → formato Markdown para preservar alineación fila-columna
            # (importante para cheatsheets con columnas Algoritmo/Ventajas/Desventajas)
            for table in (page.extract_tables() or []):
                if not table:
                    continue
                md = []
                for i, row in enumerate(table):
                    cells = [str(c).strip().replace("\n", " ") if c else "" for c in row]
                    md.append("| " + " | ".join(cells) + " |")
                    if i == 0:  # separador de encabezado Markdown
                        md.append("|" + "|".join(["---"] * len(row)) + "|")
                sections.append(f"[Página {page_num} – Tabla]\n" + "\n".join(md))
    return "\n\n".join(sections)


def chunk_text(text: str, chunk_size: int = 300, overlap: int = 50) -> list[str]:
    """Divide texto en ventanas solapadas (en palabras) para no cortar contexto."""
    words = text.split()
    step = max(1, chunk_size - overlap)  # avance entre ventanas
    return [
        " ".join(words[i: i + chunk_size])
        for i in range(0, len(words), step)
        if words[i: i + chunk_size]
    ]


# Extraer y chunkear ambos PDFs
all_chunks: list[str] = []
chunk_sources: list[str] = []

for source_name, pdf_path in PDF_PATHS.items():
    raw_text = extract_pdf_content(pdf_path)
    chunks = chunk_text(raw_text)
    all_chunks.extend(chunks)
    chunk_sources.extend([source_name] * len(chunks))
    print(f"  {source_name}: {len(chunks)} chunks extraídos")

print(f"\nTotal de chunks: {len(all_chunks)}")
print(f"\nEjemplo (primer chunk):\n{all_chunks[0][:300] if all_chunks else 'N/A'}")


  python_cheatsheet: 50 chunks extraídos
  ml_cheatsheet: 10 chunks extraídos

Total de chunks: 60

Ejemplo (primer chunk):
[Página 1] Data Types Strings Numbers & Math Real Python Python is dynamically typed It’s recommended to use double-quotes for strings Arithmetic Operators Pocket Reference Use None to represent missing or optional values Use "\n" to create a line break in a string 10 + 3 # 13 Use type() to check ob


In [29]:
# ── Embeddings + índice FAISS ─────────────────────────────────────────────────

# all-MiniLM-L6-v2: modelo ligero de 384 dimensiones, buen balance velocidad/calidad
print("Cargando modelo de embeddings all-MiniLM-L6-v2 ...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2", device=device)

print("Generando embeddings (puede tardar unos segundos) ...")
embeddings = embed_model.encode(
    all_chunks,
    convert_to_numpy=True,
    show_progress_bar=True,
    batch_size=64,
)

# IndexFlatL2: búsqueda exacta por distancia euclidiana (L2)
# Para colecciones grandes (>100k vectores) considerar IndexIVFFlat
dim = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dim)
faiss_index.add(embeddings.astype("float32"))

print(f"\nÍndice FAISS listo: {faiss_index.ntotal} vectores × {dim} dimensiones")


Cargando modelo de embeddings all-MiniLM-L6-v2 ...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8082.27it/s]


Generando embeddings (puede tardar unos segundos) ...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.41it/s]


Índice FAISS listo: 60 vectores × 384 dimensiones


In [30]:
# ── Función RAG chatbot ───────────────────────────────────────────────────────

groq_client = Groq()  # lee GROQ_API_KEY del entorno (cargada con dotenv arriba)


def rag_chatbot(question: str, k: int = 5) -> tuple[str, str]:
    """
    RAG pipeline completo:
      1. Embed la pregunta
      2. Recupera los k chunks más cercanos en FAISS (distancia L2)
      3. Inyecta el contexto en el prompt
      4. Genera respuesta con Llama 3.3 70B (Groq)

    Returns: (respuesta, contexto_recuperado)
    """
    # Paso 1: representar la pregunta como vector de 384 dim
    q_vec = embed_model.encode([question], convert_to_numpy=True).astype("float32")

    # Paso 2: buscar los k fragmentos más similares en el índice
    _, indices = faiss_index.search(q_vec, k)

    # Paso 3: armar el bloque de contexto que se inyecta al LLM
    context_parts = [
        f"[Fragmento {r} – fuente: {chunk_sources[idx]}]\n{all_chunks[idx]}"
        for r, idx in enumerate(indices[0], start=1)
    ]
    context = "\n\n".join(context_parts)

    # Paso 4: prompt con instrucción explícita de no salirse del contexto
    prompt = (
        "Eres un asistente experto en Python y Machine Learning.\n"
        "Responde la pregunta usando ÚNICAMENTE el contexto proporcionado.\n"
        "Si la información no está en el contexto, indícalo explícitamente.\n\n"
        "--- CONTEXTO ---\n"
        f"{context}\n"
        "--- FIN DEL CONTEXTO ---\n\n"
        f"Pregunta: {question}\n\n"
        "Respuesta:"
    )

    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1024,
    )

    answer = response.choices[0].message.content
    return answer, context


def llm_directo(question: str) -> str:
    """
    Llama al LLM SIN contexto RAG — sirve como baseline de validación.
    Compara su respuesta con rag_chatbot() para medir el aporte real del retrieval.
    """
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": question}],
        max_tokens=1024,
    )
    return response.choices[0].message.content


In [31]:
# ── Preguntas requeridas por la actividad (a – f) ────────────────────────────

PREGUNTAS = {
    "a": "¿Qué excepción se genera en Python al intentar dividir un número entre cero?",
    "b": "Dame al menos 2 ejemplos de métodos de strings en Python y describe qué hace cada uno.",
    "c": "¿Cuáles son las principales desventajas del algoritmo Random Forest?",
    "d": "Describe 3 casos de uso del clustering (aprendizaje no supervisado) en problemas reales de negocio.",
    "e": "¿Cuáles son los tipos de datos básicos en Python y para qué sirve cada uno?",
    "f": "¿Cuáles son las diferencias principales entre la regresión logística y los árboles de decisión?",
}

# Guardar respuestas RAG para la comparación posterior
respuestas_rag: dict[str, str] = {}

for letra, pregunta in PREGUNTAS.items():
    sep = "=" * 72
    print(f"\n{sep}")
    print(f"  PREGUNTA ({letra.upper()}): {pregunta}")
    print(sep)

    respuesta, contexto = rag_chatbot(pregunta)
    respuestas_rag[letra] = respuesta

    print(f"\nRESPUESTA RAG:\n{respuesta}")
    print(f"\n[Contexto recuperado – primeros 400 caracteres]\n{contexto[:400]}…")



  PREGUNTA (A): ¿Qué excepción se genera en Python al intentar dividir un número entre cero?

RESPUESTA RAG:
La excepción que se genera en Python al intentar dividir un número entre cero es una excepción "ZeroDivisionError". Sin embargo, en el contexto proporcionado, se menciona que al intentar dividir un número entre cero, se genera una excepción "ValueError" en el caso de que el denominador sea cero en la línea `result = 10 / number` y se capture con un bloque `except ValueError:`. Pero en realidad, en Python, intentar dividir por cero genera una "ZeroDivisionError", no una "ValueError". 

La respuesta correcta en el contexto de Python en general sería "ZeroDivisionError", pero según el contexto específico proporcionado, la excepción mencionada es "ValueError", lo que podría considerarse un error en el contexto, ya que técnicamente debería ser "ZeroDivisionError".

[Contexto recuperado – primeros 400 caracteres]
[Fragmento 1 – fuente: python_cheatsheet]
" ge"] fruits[-1] # "orange" 

In [32]:
# ── Validación de accuracy: RAG vs LLM directo ───────────────────────────────
# El mismo LLM actúa como juez: lee ambas respuestas y decide cuál es
# más precisa y específica para la pregunta. Veredicto: RAG | DIRECTO | EMPATE

def llm_directo(question: str) -> str:
    """Responde sin ningún contexto de los documentos (baseline)."""
    response = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": question}],
        max_tokens=512,
    )
    return response.choices[0].message.content


def evaluar_par(pregunta: str, resp_rag: str, resp_directa: str) -> tuple[str, str]:
    """Pide al LLM que compare ambas respuestas y devuelva veredicto + razón breve."""
    juicio_prompt = (
        f"Pregunta: {pregunta}\n\n"
        f"Respuesta A (RAG — con documentos de referencia):\n{resp_rag}\n\n"
        f"Respuesta B (LLM directo — sin documentos):\n{resp_directa}\n\n"
        "¿Cuál respuesta es más precisa, específica y correcta para la pregunta?\n"
        "Responde EXACTAMENTE en este formato:\n"
        "VEREDICTO: RAG | DIRECTO | EMPATE\n"
        "RAZÓN: (una sola oración explicando por qué)"
    )
    resp = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": juicio_prompt}],
        max_tokens=150,
        temperature=0.0,  # determinista para evaluación
    )
    texto = resp.choices[0].message.content.strip()
    # Extraer veredicto de la primera línea
    veredicto = "EMPATE"
    for linea in texto.splitlines():
        if linea.upper().startswith("VEREDICTO:"):
            v = linea.split(":", 1)[1].strip().upper()
            if "RAG" in v and "DIRECTO" not in v:
                veredicto = "RAG"
            elif "DIRECTO" in v:
                veredicto = "DIRECTO"
            break
    return veredicto, texto


# ── Ejecutar comparación ──────────────────────────────────────────────────────
resultados = []  # (letra, pregunta, veredicto, juicio_completo)

for letra, pregunta in PREGUNTAS.items():
    print(f"\n{'─' * 68}")
    print(f"({letra.upper()}) {pregunta}")

    resp_rag = respuestas_rag[letra]
    resp_directa = llm_directo(pregunta)
    veredicto, juicio = evaluar_par(pregunta, resp_rag, resp_directa)

    print(f"\n  RAG    : {resp_rag[:200]}...")
    print(f"  DIRECTO: {resp_directa[:200]}...")
    print(f"\n  {juicio}")

    resultados.append((letra, veredicto))

# ── Tabla resumen de accuracy ─────────────────────────────────────────────────
iconos = {"RAG": "✅ RAG", "DIRECTO": "❌ DIRECTO", "EMPATE": "⚠️  EMPATE"}
conteo = {"RAG": 0, "DIRECTO": 0, "EMPATE": 0}

print("\n" + "=" * 68)
print("RESUMEN DE ACCURACY")
print("=" * 68)
print(f"{'Pregunta':<12} {'Veredicto'}")
print("-" * 30)
for letra, veredicto in resultados:
    print(f"  ({letra.upper()})          {iconos[veredicto]}")
    conteo[veredicto] += 1

total = len(resultados)
print("-" * 30)
print(f"  RAG gana   : {conteo['RAG']}/{total} preguntas")
print(f"  LLM directo: {conteo['DIRECTO']}/{total} preguntas")
print(f"  Empate     : {conteo['EMPATE']}/{total} preguntas")
print(f"\n  Accuracy RAG: {conteo['RAG']/total*100:.0f}% de las preguntas")



────────────────────────────────────────────────────────────────────
(A) ¿Qué excepción se genera en Python al intentar dividir un número entre cero?

  RAG    : La excepción que se genera en Python al intentar dividir un número entre cero es una excepción "ZeroDivisionError". Sin embargo, en el contexto proporcionado, se menciona que al intentar dividir un nú...
  DIRECTO: La excepción que se genera en Python al intentar dividir un número entre cero es `ZeroDivisionError`. Esta excepción se lanza cuando se intenta realizar una operación de división o módulo con un divis...

  VEREDICTO: DIRECTO
RAZÓN: La respuesta directa proporciona la información correcta y precisa sobre la excepción que se genera en Python al intentar dividir un número entre cero, que es "ZeroDivisionError", sin confusión ni referencia a un contexto incorrecto.

────────────────────────────────────────────────────────────────────
(B) Dame al menos 2 ejemplos de métodos de strings en Python y describe qué hace cada

# **Conclusiones:**

* #### **Conclusiones de la actividad chatbot LLM + RAG para documentos con información tabular:**

---

### 1. Desafíos de la extracción de información tabular en PDFs visuales

Los cheatsheets usados (`python_cheatsheet.pdf` y `ml_cheatsheet.pdf`) son PDFs de tipo infografía con diseño multi-columna. Esto genera los siguientes retos:

- **Orden de lectura incorrecto:** `pdfplumber` extrae texto de forma lineal (izquierda → derecha, arriba → abajo), mezclando columnas que semánticamente no están relacionadas.
- **Pérdida de estructura tabular:** columnas como `Algoritmo | Descripción | Ventajas | Desventajas` se fragmentan porque los valores de múltiples celdas se concatenan sin separación clara.
- **Ruido por elementos visuales:** iconos, colores de fondo y gráficos no se traducen a texto, dejando huecos en el contenido extraído.
- **Contexto semántico difuso:** al chunkear, un mismo fragmento puede mezclar partes de diferentes algoritmos o tópicos, reduciendo la precisión del retrieval.

### 2. Soluciones aplicadas

- Extracción combinada: texto plano + `extract_tables()` con conversión a Markdown para preservar estructura fila-columna.
- Chunking con solapamiento (overlap=50 palabras) para no cortar contexto importante en los bordes de fragmentos.
- Prompt engineering que instruye a Claude a responder **solo** con el contexto recuperado, minimizando alucinaciones.

### 3. Reflexión sobre escalabilidad a producción

Para un sistema RAG productivo con documentos tabulares complejos se recomienda:

1. **Extracción especializada:** `camelot`, `tabula-py` o `unstructured.io` para mayor fidelidad en tablas estructuradas.
2. **Procesamiento multimodal:** modelos que analicen el PDF como imagen (Claude con visión) para preservar layout visual.
3. **Metadata enriquecida:** indexar fuente, número de página y tipo de contenido junto con el embedding para filtrado más preciso.
4. **Evaluación de retrieval:** métricas como MRR y nDCG para medir y mejorar la calidad de la recuperación vectorial.

### 4. Resultado

El sistema RAG implementado respondió las 6 preguntas requeridas, demostrando que incluso con PDFs visuales complejos, la combinación de extracción híbrida (texto + tablas Markdown) + FAISS + Llama 3.3 70B (Groq) produce un chatbot funcional y preciso.


### 5. ¿Por qué el LLM directo puede ganar a RAG en este ejercicio?

La validación comparativa puede mostrar que el LLM directo responde igual o mejor que RAG en varias preguntas. Esto **no es un fallo del pipeline** — es un fenómeno esperado y tiene dos causas:

#### Causa 1: el conocimiento ya está en el pretraining

Los documentos usados (`python_cheatsheet.pdf`, `ml_cheatsheet.pdf`) cubren temas que Llama 3.3 70B ya memorizó durante su entrenamiento: excepciones de Python, métodos de strings, algoritmos de ML, etc. Son conceptos públicos y ampliamente documentados en internet.

> RAG aporta valor cuando la respuesta **solo existe en el documento** (datos privados, versiones específicas, tablas propietarias). Si el LLM ya "sabe" la respuesta, el contexto recuperado es redundante o incluso ruido.

#### Causa 2: gap semántico español ↔ inglés

Las preguntas están en **español**, pero los PDFs están en **inglés**. El modelo de embeddings `all-MiniLM-L6-v2` fue entrenado principalmente en inglés, por lo que la representación vectorial de una pregunta en español y su respuesta en inglés no son tan cercanas como deberían.

```
Pregunta (ES): "¿Qué excepción se genera al dividir entre cero?"
Chunk relevante (EN): "ZeroDivisionError: division by zero"
→ Distancia L2 más alta de lo esperado → puede que no llegue al top-5
```

Esto hace que FAISS recupere fragmentos menos relevantes, y el LLM genere una respuesta pobre o diga "no está en el contexto".

---

### 6. Cómo corregirlo

| Problema | Solución | Cambio en código |
| --- | --- | --- |
| Gap español↔inglés en embeddings | Modelo multilingüe | `SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")` |
| Prompt demasiado restrictivo | Permitir conocimiento propio si el contexto es débil | Cambiar `"ÚNICAMENTE el contexto"` → `"Prioriza el contexto; si no es suficiente, usa tu conocimiento"` |
| Chunks con contexto mezclado | Chunking semántico por párrafo/sección | Dividir por `\n\n` (párrafos) en lugar de ventana fija de palabras |
| Retrieval por distancia L2 simple | Hybrid search: BM25 + FAISS | Combinar similitud semántica con coincidencia de palabras clave |
| Chunks ruidosos por PDF infografía | Extracción multimodal | Convertir páginas a imagen y procesarlas con un LLM con visión |

> **Conclusión clave:** RAG es la arquitectura correcta para documentos privados o información que el LLM no conoce. Para documentos de conocimiento general, el beneficio es marginal y depende críticamente de la calidad de la extracción y del modelo de embeddings elegido.


# **Fin de la actividad chatbot: LLM + RAG**